# SD3 Token Attention — layer × timestep

각 transformer block의 image→text token attention을 **모든 28 step**에 걸쳐 캡처해
**timestep별로 1개씩 총 28개 figure**를 출력.

각 figure: row = block (0~23), col = prompt token.

In [ ]:
import math
import torch
import numpy as np
import matplotlib.pyplot as plt

from diffusers import StableDiffusion3Pipeline


# =========================================================
# 설정
# =========================================================
MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"

PROMPT = "The Colosseum Rome Italy Carry-all Pouch"

NUM_INFERENCE_STEPS = 28
GUIDANCE_SCALE      = 1.0

SEED = 0

DEVICE = "cuda"
DTYPE  = torch.float16

# None이면 모든 valid token 표시. 예: ["red", "car", "road"]
TARGET_TOKEN_STRINGS = None

/home/haeun/miniconda3/envs/C3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Attention 저장소

`(step, block_idx)` → `(image_tokens, text_tokens)` tensor.
기존 코드는 `current_step() != 0` 일 때 skip 했지만, 여기서는 **모든 step을 저장**.

In [2]:
class SD3AttentionStore:
    """
    block 호출 순서로부터 step을 역산:
      step = call_idx // num_blocks
    (CFG가 어떻든 transformer.forward 1회당 모든 block이 한 번씩 호출됨)

    저장 형식:
      maps[(step, block_idx)] = tensor [image_tokens, text_tokens]  (head/batch 평균)
    """

    def __init__(self, num_blocks):
        self.num_blocks = num_blocks
        self.call_idx   = 0
        self.maps       = {}

    def current_step(self):
        return self.call_idx // self.num_blocks

    def save(self, block_idx, image_to_text_attn):
        # image_to_text_attn: [B, H, image_tokens, text_tokens]
        attn = image_to_text_attn.detach().float().cpu()
        attn = attn.mean(dim=0).mean(dim=0)   # B, H 평균 → [image_tokens, text_tokens]
        self.maps[(self.current_step(), block_idx)] = attn

    def step_call_end(self):
        self.call_idx += 1

    def num_steps_captured(self):
        if not self.maps:
            return 0
        return max(s for s, _ in self.maps.keys()) + 1

## SD3 Joint Attention Processor

diffusers의 SD3 joint attention을 mirror하면서 image→text attention을 추출.
(joint sequence: `[text(text_len), image(img_len)]`)

In [ ]:
class StoreSD3JointAttnProcessor:
    """
    diffusers의 JointAttnProcessor2_0 mirror — attention prob 캡처용.

    중요:
      - SD3.5 medium은 attention 안에 qk_norm(RMSNorm)이 있음
        → norm_q / norm_k / norm_added_q / norm_added_k 반드시 적용해야 정상 generation
      - fp16 softmax는 overflow 위험 → fp32로 변환 후 softmax
      - context_pre_only=True (block 23) 은 to_add_out 이 None
    """

    def __init__(self, store, block_idx):
        self.store     = store
        self.block_idx = block_idx

    def __call__(
        self,
        attn,
        hidden_states,
        encoder_hidden_states=None,
        attention_mask=None,
        **kwargs,
    ):
        batch_size = hidden_states.shape[0]
        img_len    = hidden_states.shape[1]
        text_len   = encoder_hidden_states.shape[1]

        # ── image QKV ──────────────────────────────────────────
        query = attn.to_q(hidden_states)
        key   = attn.to_k(hidden_states)
        value = attn.to_v(hidden_states)

        inner_dim = key.shape[-1]
        head_dim  = inner_dim // attn.heads

        query = query.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)
        key   = key.view(batch_size,   -1, attn.heads, head_dim).transpose(1, 2)
        value = value.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)

        # qk_norm (SD3.5)
        if getattr(attn, "norm_q", None) is not None:
            query = attn.norm_q(query)
        if getattr(attn, "norm_k", None) is not None:
            key = attn.norm_k(key)

        # ── text QKV ───────────────────────────────────────────
        encoder_query = attn.add_q_proj(encoder_hidden_states)
        encoder_key   = attn.add_k_proj(encoder_hidden_states)
        encoder_value = attn.add_v_proj(encoder_hidden_states)

        encoder_query = encoder_query.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)
        encoder_key   = encoder_key.view(batch_size,   -1, attn.heads, head_dim).transpose(1, 2)
        encoder_value = encoder_value.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)

        # added qk_norm (SD3.5)
        if getattr(attn, "norm_added_q", None) is not None:
            encoder_query = attn.norm_added_q(encoder_query)
        if getattr(attn, "norm_added_k", None) is not None:
            encoder_key = attn.norm_added_k(encoder_key)

        # ── joint attention: [text, image] 순으로 concat ───────
        query = torch.cat([encoder_query, query], dim=2)
        key   = torch.cat([encoder_key,   key],   dim=2)
        value = torch.cat([encoder_value, value], dim=2)

        # ── attention (fp32 softmax for stability) ─────────────
        scale       = 1.0 / math.sqrt(head_dim)
        attn_scores = torch.matmul(query, key.transpose(-1, -2)) * scale
        attn_probs  = torch.softmax(attn_scores.float(), dim=-1).to(query.dtype)

        # ── image → text 영역만 추출 ──────────────────────────
        # row: image queries (= [text_len : text_len + img_len])
        # col: text keys     (= [: text_len])
        image_to_text = attn_probs[
            :,
            :,
            text_len : text_len + img_len,
            : text_len,
        ]
        self.store.save(self.block_idx, image_to_text)

        # ── 실제 forward output ────────────────────────────────
        hidden_states = torch.matmul(attn_probs, value)
        hidden_states = hidden_states.transpose(1, 2).reshape(
            batch_size,
            text_len + img_len,
            attn.heads * head_dim,
        )

        encoder_hidden_states = hidden_states[:, :text_len]
        hidden_states         = hidden_states[:, text_len:]

        hidden_states = attn.to_out[0](hidden_states)
        hidden_states = attn.to_out[1](hidden_states)

        # context_pre_only=True (block 23) 이면 to_add_out 이 없음 → 그대로 반환
        # JointTransformerBlock 이 어차피 context_attn_output 을 무시함
        if not getattr(attn, "context_pre_only", False):
            encoder_hidden_states = attn.to_add_out(encoder_hidden_states)

        self.store.step_call_end()

        return hidden_states, encoder_hidden_states

## Token 처리 헬퍼

In [ ]:
def _clean(tok):
    # SentencePiece(▁) / BPE(</w>) 표식 제거 — 시각화 라벨용
    return tok.replace("▁", "").replace("</w>", "")


def get_tokens(pipe, prompt):
    tokenizer = pipe.tokenizer_3
    tokenized = tokenizer(
        prompt,
        padding       = "max_length",
        max_length    = pipe.tokenizer_max_length,
        truncation    = True,
        return_tensors= "pt",
    )
    input_ids = tokenized.input_ids[0]
    tokens    = tokenizer.convert_ids_to_tokens(input_ids)

    valid = []
    for idx, tok in enumerate(tokens):
        if tok in tokenizer.all_special_tokens:
            continue
        if tok == tokenizer.pad_token:
            continue
        if tok == "<pad>":
            continue
        valid.append((idx, _clean(tok)))   # ← 정리된 라벨만 저장
    return valid


def filter_tokens(valid_tokens):
    if TARGET_TOKEN_STRINGS is None:
        return valid_tokens
    selected = []
    for idx, tok in valid_tokens:
        for target in TARGET_TOKEN_STRINGS:
            if target.lower() in tok.lower():
                selected.append((idx, tok))
                break
    return selected

## 시각화 — 선그래프 (mean activation score)

block을 x축, 각 token의 image→text mean attention을 y축으로 plot.
일단 **step 0** 만 본다 (다른 step도 보고 싶으면 `STEP_TO_PLOT` 변경).

In [ ]:
def _distinct_styles(n):
    """
    n개의 (color, linestyle) 조합을 절대 겹치지 않게 생성.
      - color: HSV 색공간에서 균등 분할 → n개 distinct hue
      - linestyle: ["-", "--", "-.", ":"] 를 4개 단위로 cycle
        (color 자체가 모두 다르지만, 인접 hue 구분 보조용)
    """
    cmap     = plt.cm.hsv
    colors   = [cmap(i / max(n, 1)) for i in range(n)]
    ls_pool  = ["-", "--", "-.", ":"]
    styles   = [(colors[i], ls_pool[i % len(ls_pool)]) for i in range(n)]
    return styles


def plot_mean_activation_line(store, selected_tokens, step=0):
    """
    block을 x축으로 하는 선그래프.
      x = block_idx (0..num_blocks-1)
      y = mean image→text attention  (image_tokens 차원에 대해 평균)
      line = 각 prompt token (모두 색깔 distinct)
    """
    block_indices = sorted(b for (s, b) in store.maps.keys() if s == step)
    if not block_indices:
        print(f"[WARN] step {step} 캡처 없음")
        return

    token_curves = {tok_str: [] for _, tok_str in selected_tokens}
    x_vals       = []

    for block_idx in block_indices:
        block_attn     = store.maps[(step, block_idx)]      # [img_tokens, text_tokens]
        per_token_mean = block_attn.mean(dim=0).numpy()     # [text_tokens]
        x_vals.append(block_idx)
        for tok_idx, tok_str in selected_tokens:
            if tok_idx < per_token_mean.shape[0]:
                token_curves[tok_str].append(float(per_token_mean[tok_idx]))
            else:
                token_curves[tok_str].append(np.nan)

    styles = _distinct_styles(len(selected_tokens))

    fig, ax = plt.subplots(figsize=(11, 5))
    for (tok_str, ys), (color, ls) in zip(token_curves.items(), styles):
        ax.plot(
            x_vals, ys,
            color     = color,
            linestyle = ls,
            marker    = "o",
            linewidth = 1.6,
            markersize= 4,
            label     = tok_str,
        )

    ax.set_xlabel("Block index")
    ax.set_ylabel("Mean image → text attention")
    ax.set_title(f"Mean activation score per block  |  step {step}  |  prompt: \"{PROMPT}\"")
    ax.set_xticks(x_vals)
    ax.grid(True, alpha=0.3)
    # legend 는 plot 밖 오른쪽으로 — 색이 많아도 가독성 유지
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=9, frameon=False)
    fig.tight_layout()
    plt.show()

## 실행

In [ ]:
torch.set_grad_enabled(False)

# ── 파이프라인 로드 (재사용) ─────────────────────────────────
if "pipe" not in dir():
    pipe = StableDiffusion3Pipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE)
pipe.set_progress_bar_config(disable=True)

num_blocks = len(pipe.transformer.transformer_blocks)
print(f"[INFO] num_blocks = {num_blocks}")

# ── store 초기화 + processor 부착 ──────────────────────────
store = SD3AttentionStore(num_blocks)
for block_idx, block in enumerate(pipe.transformer.transformer_blocks):
    block.attn.set_processor(
        StoreSD3JointAttnProcessor(store=store, block_idx=block_idx)
    )

# ── token 추출 ─────────────────────────────────────────────
valid_tokens    = get_tokens(pipe, PROMPT)
selected_tokens = filter_tokens(valid_tokens)

print("[INFO] Selected tokens:")
for idx, tok in selected_tokens:
    print(f"  {idx:>3}  {tok}")

# ── generation ─────────────────────────────────────────────
generator = torch.Generator(device=DEVICE).manual_seed(SEED)
result = pipe(
    prompt              = PROMPT,
    num_inference_steps = NUM_INFERENCE_STEPS,
    guidance_scale      = GUIDANCE_SCALE,
    generator           = generator,
)

n_steps = store.num_steps_captured()
print(f"[INFO] captured steps = {n_steps}")
print(f"[INFO] captured pairs = {len(store.maps)}  (예상: {n_steps * num_blocks})")

result.images[0]

Loading pipeline components...:   0%|          | 0/9 [00:01<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# step 0 만 — 각 token의 mean activation score 선그래프
STEP_TO_PLOT = 0
plot_mean_activation_line(
    store           = store,
    selected_tokens = selected_tokens,
    step            = STEP_TO_PLOT,
)